In [50]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [51]:
air_quality_data = pd.read_csv('/content/week3_air_quality_hourly_20260305_145306.csv')

In [52]:
x = air_quality_data.drop(columns=['city', 'state', 'zip', 'time', 'us_aqi'])
y = air_quality_data['us_aqi']

In [53]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

In [54]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1,
    shuffle=True,
    random_state=42
)

In [55]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)

In [56]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1,1)
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.float32).view(-1,1)

In [57]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

In [58]:
class Linear_Autoencoder(nn.Module):
  def __init__(self, input_dims):
    super().__init__()
    self.encoder = nn.Sequential(
        nn.Linear(input_dims, 8),
        nn.Linear(8, 4),
    )

    self.decoder = nn.Sequential(
        nn.Linear(4, 8),
        nn.Linear(8, input_dims)
    )

  def forward(self, x):
    encode = self.encoder(x)
    decode = self.decoder(encode)
    return decode


In [59]:
model = Linear_Autoencoder(X_train_tensor.shape[1])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [60]:
epochs = 20

for i in range(epochs):
  sum_batch = 0
  model.train()

  for X_batch, _ in train_loader:
    reconstruction = model(X_batch)

    loss = criterion(reconstruction, X_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    sum_batch = sum_batch + loss.item()

  model.eval()
  with torch.no_grad():
    val_reconstruction = model(X_val_tensor)
    val_loss = criterion(val_reconstruction, X_val_tensor)
  print(f'Epoch {i+1}: Loss: train={(sum_batch / len(train_loader)):.4f}, val={val_loss.item():.4f}')


Epoch 1: Loss: train=0.9890, val=0.8503
Epoch 2: Loss: train=0.7334, val=0.5873
Epoch 3: Loss: train=0.4864, val=0.3990
Epoch 4: Loss: train=0.3840, val=0.3617
Epoch 5: Loss: train=0.3430, val=0.3106
Epoch 6: Loss: train=0.2756, val=0.2405
Epoch 7: Loss: train=0.2230, val=0.2044
Epoch 8: Loss: train=0.1949, val=0.1811
Epoch 9: Loss: train=0.1739, val=0.1632
Epoch 10: Loss: train=0.1586, val=0.1512
Epoch 11: Loss: train=0.1475, val=0.1422
Epoch 12: Loss: train=0.1392, val=0.1349
Epoch 13: Loss: train=0.1318, val=0.1292
Epoch 14: Loss: train=0.1265, val=0.1252
Epoch 15: Loss: train=0.1233, val=0.1226
Epoch 16: Loss: train=0.1212, val=0.1210
Epoch 17: Loss: train=0.1195, val=0.1199
Epoch 18: Loss: train=0.1187, val=0.1194
Epoch 19: Loss: train=0.1184, val=0.1191
Epoch 20: Loss: train=0.1185, val=0.1188


In [61]:
model.eval()

with torch.no_grad():
  test_recon = model(X_test_tensor)
  test_loss = criterion(test_recon, X_test_tensor)
print(f'Test Loss={test_loss.item():.4f}')

Test Loss=0.1178
